In [1]:
import numpy as np
import matplotlib.pyplot as plt
import scipy 
from pfapack import pfaffian as pf


We assume the form 

$$X = e^{\frac{1}{4}\gamma^T A \gamma}$$

which is characterized by 

$$\eta, B$$

In [ ]:
def get_eta_B(A):

    B = scipy.linalg.expm(-A)

    sinhA = scipy.linalg.sinhm(A/4)

    bdim = B.shape[0]

    M = np.zeros((2*bdim, 2*bdim), dtype = np.complex128)

    M[0:bdim, 0:bdim] = np.sqrt(2) * sinhA
    M[bdim:,0:bdim]   = np.identity(bdim)
    M[0:bdim,bdim:]   = -np.identity(bdim)
    M[bdim:,bdim:]    = np.sqrt(2) * sinhA

    eta = np.power(-1,bdim//2) * pf.pfaffian(M)

    return eta, B

def multiplication(eta1, B1, eta2, B2):

    bdim = B1.shape[0]

    M = np.zeros((2*bdim, 2*bdim), dtype = np.complex128)

    G1 = (np.identity(bdim) - B1) @ scipy.linalg.inv(np.identity(bdim) + B1)
    G2 = (np.identity(bdim) - B2) @ scipy.linalg.inv(np.identity(bdim) + B2)

    M[0:bdim, 0:bdim] = G1
    M[bdim:,0:bdim]   = np.identity(bdim)
    M[0:bdim,bdim:]   = -np.identity(bdim)
    M[bdim:,bdim:]    = G2

    eta12 = np.power(-1,bdim//2) * eta1 * eta2 * pf.pfaffian(M)

    B12   = B1@B2

    return eta12, B12

def recursive_eta(Alist):

    Ntau = len(Alist)
    N    = Alist[0].shape[0]//2

    eta, B = get_eta_B(Alist[0])

    eta_prod = eta
    B_prod = B

    for i in range(Ntau-1):

        next_eta, next_B = get_eta_B(Alist[i+1])

        eta_prod, B_prod = multiplication(eta_prod, B_prod, next_eta, next_B)

    return eta_prod*np.power(2, N)

def brute_force_trace(Alist):

    Ntau = len(Alist)
    N    = Alist[0].shape[0] // 2


    eta_prod = 1
    etas = np.zeros(Ntau, dtype=np.complex128)
    Bs   = np.zeros((2*N, 2*N, Ntau), dtype=np.complex128)
    Gs   = np.zeros((2*N, 2*N, Ntau), dtype=np.complex128)

    M    = np.zeros((2*N*Ntau, 2*N*Ntau), dtype=np.complex128)

    I = np.eye(2*N, dtype=np.complex128)

    # Fill M in 2N x 2N blocks
    for i in range(Ntau):
        for j in range(i + 1, Ntau):
            # checkerboard on the upper-right triangle, starting with "-"
            sgn = -1 if ((i + j) % 2 == 1) else 1

            r_i = slice(2*N*i, 2*N*(i + 1))
            r_j = slice(2*N*j, 2*N*(j + 1))

            M[r_i, r_j] = sgn * I          # upper-right block
            M[r_j, r_i] = -sgn * I         # lower-left block (antisymmetric)


    for i in range(Ntau):
        etas[i], Bs[:, :, i] = get_eta_B(Alist[i])
        Gs[:, :, i]          = scipy.linalg.tanhm(Alist[i] / 2)
        eta_prod *= etas[i]
        M[i*2*N:(i+1)*2*N,i*2*N:(i+1)*2*N] = Gs[:,:,i]

    W = np.power(-1, N*Ntau//2) * np.power(2, N) * eta_prod * pf.pfaffian(M)

    return W

def catterpillar(Alist):

    Ntau = len(Alist)
    N    = Alist[0].shape[0] // 2

    temp = np.zeros((4*N, 4*N), dtype = np.complex128)

    Bs = np.zeros((2*N, 2*N, Ntau), dtype = np.complex128)
    etas = np.zeros(Ntau, dtype = np.complex128)
    eta_prod = 1
    pf_prod  = 1

    current_Bstring = np.identity(2*N, dtype=np.complex128)

    for i in range(Ntau):

        etas[i], Bs[:,:,i] = get_eta_B(Alist[i])
        eta_prod *= etas[i]

    for i in range(Ntau - 1):

        current_Bstring = current_Bstring @ Bs[:,:,i]

        temp[:2*N,:2*N] = (np.identity(2*N) - current_Bstring) @ scipy.linalg.inv(np.identity(2*N) + current_Bstring)
        temp[:2*N,2*N:] = -np.identity(2*N)
        temp[2*N:,:2*N] = np.identity(2*N)
        temp[2*N:,2*N:] = scipy.linalg.tanhm(0.5*Alist[i+1])

        pf_prod *= pf.pfaffian(temp)

    W = np.power(2, N) * np.power(-1, (Ntau + 1)*N) *eta_prod * pf_prod

    return W

def generate_random_Alist(N, Ntau, scaling = 0.1):

    Alist = []
    rng = np.random.default_rng(seed = 412425)

    for i in range(Ntau):

        A = rng.normal(loc = 0, scale = scaling, size = (2*N, 2*N)) + 1j * rng.normal(loc = 0, scale = scaling, size = (2*N, 2*N))
        A = 0.5*(A - A.T)
        Alist.append(A)

    return Alist

def get_J_cross(matrix):

    dim = matrix.shape[0]

    Jcross = np.zeros((2*dim, 2*dim), dtype = matrix.dtype)

    Jcross[:dim, dim:2*dim] = - matrix
    Jcross[dim:2*dim, :dim] = + matrix

    return Jcross

In [28]:
phi = np.diag(np.array((0.241,415.24, 14, 13)))

k = get_J_cross(phi)

print(k)

None


In [21]:
catterpillar(Alist)

np.complex128(3.9925372061292452+0.012124206907000286j)

In [22]:
brute_force_trace(Alist)

np.complex128(3.9925372061292452+0.012124206907000288j)

In [23]:
recursive_eta(Alist)

np.complex128(3.9925372061292452+0.012124206907000293j)